In [1]:
# 1. Install core dependencies
!pip install -q torch==2.4.1 torchvision --extra-index-url https://download.pytorch.org/whl/cu121
!pip install -U "transformers<4.46.0" "peft>=0.12.0" "accelerate>=0.33.0"

# 2. Clone and setup OmniGen
!git clone https://github.com/VectorSpaceLab/OmniGen.git
%cd /content/OmniGen
!sed -i 's/accelerate==0.26.1/accelerate>=0.26.1/' setup.py
!pip install -e .

# 3. FORCE FIX: Downgrade diffusers AFTER OmniGen installation to override its requirements
!pip install -U "diffusers==0.30.0"

# 4. Install UI dependencies
!pip install -q gradio spaces

print("\n✅ Setup finished. Please restart the session if you still see import errors, then run the UI cell.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.9/798.9 MB 771.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 83.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 56.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 104.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 8.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:

import gradio as gr
from OmniGen import OmniGenPipeline

# ============================================================
# LOAD MODEL
# ============================================================
print("⏳ Đang load OmniGen-v1...")
# Removed 'offload_model=True' as it is no longer a valid argument
pipe = OmniGenPipeline.from_pretrained(
    "Shitao/OmniGen-v1",
)
# Manually move to GPU to ensure it uses CUDA
pipe.to("cuda")
print("✅ OmniGen-v1 đã sẵn sàng!")

# ============================================================
# GRADIO INTERFACE
# ============================================================
def generate_image(
    prompt,
    input_images,
    height,
    width,
    guidance_scale,
    img_guidance_scale,
    seed,
    max_input_image_size,
):
    """Tạo/chỉnh sửa ảnh bằng OmniGen"""
    # Xử lý input images
    img_list = []
    if input_images:
        for img in input_images:
            if isinstance(img, dict):
                img_list.append(img["path"])
            else:
                img_list.append(img)

    # Kiểm tra placeholder
    placeholder_count = prompt.count("<|image_")
    if len(img_list) == 0 and placeholder_count > 0:
        return None, "❌ Bạn có placeholder ảnh nhưng chưa upload ảnh!"
    if len(img_list) > 0 and placeholder_count == 0:
        # Auto thêm placeholder nếu thiếu
        for i in range(len(img_list)):
            prompt += f" <img><|image_{i+1}|></img>"

    try:
        images = pipe(
            prompt=prompt,
            input_images=img_list if img_list else None,
            height=int(height),
            width=int(width),
            guidance_scale=guidance_scale,
            img_guidance_scale=img_guidance_scale,
            seed=int(seed),
            max_input_image_size=int(max_input_image_size),
            separate_cfg_infer=True,  # Tiết kiệm VRAM trên T4
            use_kv_cache=False,
        )
        return images[0], "✅ Thành công!"
    except Exception as e:
        return None, f"❌ Lỗi: {str(e)}"


# Giao diện
with gr.Blocks(
    title="🎨 OmniGen - Uncensored AI Image Editor",
    theme=gr.themes.Soft(primary_hue="purple"),
) as demo:

    gr.Markdown("""
    # 🎨 OmniGen — Uncensored AI Image Editor
    Model thống nhất: Text→Image • Image Editing • Subject-Driven • Identity-Preserving

    **Cách dùng:** Gõ lệnh mô tả. Nếu muốn chỉnh sửa ảnh, upload ảnh và dùng `<img><|image_1|></img>` để tham chiếu ảnh trong prompt.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(
                label="✍️ Prompt",
                placeholder="VD: A curly-haired man in a red shirt is drinking tea.\n\nVới ảnh: A man in a black shirt is reading a book. The man is the right man in <img><|image_1|></img>.",
                lines=4,
            )
            input_images = gr.File(
                label="📸 Ảnh đầu vào (có thể nhiều ảnh)",
                file_count="multiple",
                file_types=["image"],
                type="filepath",
            )

            with gr.Row():
                height = gr.Slider(512, 1536, value=1024, step=64, label="Chiều cao")
                width = gr.Slider(512, 1536, value=1024, step=64, label="Chiều rộng")

            with gr.Accordion("⚙️ Cài đặt nâng cao", open=False):
                guidance_scale = gr.Slider(1.0, 5.0, value=2.5, step=0.1,
                    label="Guidance Scale (cao = theo prompt hơn)")
                img_guidance_scale = gr.Slider(1.0, 3.0, value=1.6, step=0.1,
                    label="Image Guidance Scale (cao = giữ ảnh gốc hơn)")
                seed = gr.Slider(-1, 999999, value=-1, step=1, label="Seed (-1 = random)")
                max_input_image_size = gr.Slider(256, 1024, value=1024, step=64,
                    label="Max Input Image Size (giảm nếu OOM)")

            with gr.Row():
                run_btn = gr.Button("🚀 Tạo ảnh", variant="primary", size="lg")
                clear_btn = gr.Button("🗑️ Xóa")

            gr.Examples(
                examples=[
                    ["A cyberpunk city at night with neon lights reflecting on wet streets", None, 1024, 1024, 2.5, 1.6, -1, 1024],
                    ["A watercolor painting of a peaceful Japanese garden with cherry blossoms", None, 1024, 1024, 2.5, 1.6, -1, 1024],
                    ["Turn the person in <img><|image_1|></img> into a Viking warrior", None, 1024, 1024, 2.5, 1.6, -1, 1024],
                    ["Replace the background of <img><|image_1|></img> with a tropical beach sunset", None, 1024, 1024, 2.5, 1.6, -1, 1024],
                    ["Make <img><|image_1|></img> look like an oil painting by Van Gogh", None, 1024, 1024, 2.5, 1.6, -1, 1024],
                ],
                inputs=[prompt, input_images, height, width, guidance_scale, img_guidance_scale, seed, max_input_image_size],
            )

        with gr.Column(scale=1):
            output_image = gr.Image(type="pil", label="🖼️ Kết quả")
            status = gr.Textbox(label="Trạng thái", interactive=False)

    # Event handlers
    run_btn.click(
        fn=generate_image,
        inputs=[prompt, input_images, height, width, guidance_scale, img_guidance_scale, seed, max_input_image_size],
        outputs=[output_image, status],
    )

    clear_btn.click(
        fn=lambda: ("", None, 1024, 1024, 2.5, 1.6, -1, 1024, None, ""),
        inputs=[],
        outputs=[prompt, input_images, height, width, guidance_scale, img_guidance_scale, seed, max_input_image_size, output_image, status],
    )

# ============================================================
# LAUNCH — Public URL qua gradio.live
# ============================================================
demo.launch(share=True)

⏳ Đang load OmniGen-v1...
Model not found, downloading...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Downloaded model to /root/.cache/huggingface/hub/models--Shitao--OmniGen-v1/snapshots/58e249c7c7634423c0ba41c34a774af79aa87889
